In [ ]:
import json
import uuid

import numpy as np
import psycopg2
import sklearn

from psycopg2.extras import Json
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from cobertura_common import (
    ENTRADAS,
    leer_bytes,
    leer_objeto,
    guardar_objeto,
    guardar_bytes,
    asegurar_bucket,
)

manifiesto = json.loads(leer_bytes("grupo3/latest.json"))
paquete_anterior = leer_objeto(manifiesto["objeto"])

# Permite volver a ejecutar el notebook sobre la misma instantánea.
run_origen = paquete_anterior.get(
    "run_id_datos", paquete_anterior["run_id"]
)

db = psycopg2.connect()  # Lee las variables PG* del contenedor.
try:
    with db.cursor() as cursor:
        cursor.execute("""
            SELECT features, etiqueta, conjunto
            FROM cobertura_ready
            WHERE run_id = %s
            ORDER BY row_id
        """, (run_origen,))
        filas = cursor.fetchall()
finally:
    db.close()

if not filas:
    raise ValueError(
        f"No hay datos preparados para la ejecución {run_origen}"
    )

X = np.asarray([fila[0] for fila in filas], dtype=float)
y = np.asarray([fila[1] for fila in filas], dtype=int)
train = np.asarray([fila[2] == "train" for fila in filas])
test = ~train

# Este preprocesador corresponde a las variables ya transformadas.
preprocesador = paquete_anterior["preprocesador"]

print("Ejecución de origen:", run_origen)
print("Entrenamiento:", train.sum())
print("Prueba:", test.sum())
print("Variables transformadas:", X.shape[1])

In [ ]:
modelo = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=2,
)

modelo.fit(X[train], y[train])
predicciones = modelo.predict(X[test])

metricas = {
    "accuracy": float(accuracy_score(y[test], predicciones)),
    "f1_macro": float(f1_score(
        y[test],
        predicciones,
        average="macro",
        zero_division=0,
    )),
    "filas_train": int(train.sum()),
    "filas_test": int(test.sum()),
    "batches_recolectados": (
        paquete_anterior["metricas"]["batches_recolectados"]
    ),
    "clases_entrenadas": modelo.classes_.tolist(),
    "clases_test": np.unique(y[test]).tolist(),
    "matriz_confusion_orden_1_a_7": confusion_matrix(
        y[test], predicciones, labels=list(range(1, 8))
    ).tolist(),
}

print(json.dumps(metricas, indent=2))

In [ ]:
asegurar_bucket()

version = "jupyter_" + uuid.uuid4().hex
carpeta = f"grupo3/{version}"
objeto = f"{carpeta}/modelo.joblib"

paquete = {
    "modelo": modelo,
    "preprocesador": preprocesador,
    "columnas": ENTRADAS,
    "metricas": metricas,
    "run_id": version,
    "run_id_datos": run_origen,
    "sklearn_version": sklearn.__version__,
}

checksum = guardar_objeto(objeto, paquete)

guardar_bytes(
    f"{carpeta}/metricas.json",
    json.dumps(metricas).encode(),
    "application/json",
)

db = psycopg2.connect()
try:
    with db:
        with db.cursor() as cursor:
            cursor.execute("""
                INSERT INTO cobertura_modelos
                    (run_id, objeto, metricas)
                VALUES (%s, %s, %s)
            """, (version, objeto, Json(metricas)))
finally:
    db.close()

print("Modelo guardado:", objeto)

In [ ]:
nuevo_manifiesto = {
    "objeto": objeto,
    "sha256": checksum,
    "run_id": version,
    "metricas": metricas,
}

guardar_bytes(
    "grupo3/latest.json",
    json.dumps(nuevo_manifiesto).encode(),
    "application/json",
)

print("Modelo activado:", version)